# Interactive AXI GPIO & LED Control via PYNQ MMIO
Use PYNQ's `MMIO` class to interact directly with hardware registers mapped to an AXI GPIO core in your Programmable Logic (PL).

### Hardware Pin Mapping (Master XDC):
- **D5 Red LED**: Pin `M19` (Bank 35, 3.3V LVCMOS)
- **D6 Green LED**: Pin `M17` (Bank 35, 3.3V LVCMOS)
- **AXI GPIO Base Address**: `0x41200000` (assigned in Vivado Address Editor)



In [ ]:
import time
from pynq import MMIO
import ipywidgets as widgets
from IPython.display import display

# Standard Xilinx AXI GPIO Register Map
AXI_GPIO_BASE = 0x41200000
ADDRESS_RANGE = 0x1000

GPIO_DATA = 0x0000  # Channel 1 Data Register
GPIO_TRI  = 0x0004  # Channel 1 Tri-state Register (0 = Output, 1 = Input)

try:
    gpio = MMIO(AXI_GPIO_BASE, ADDRESS_RANGE)
    # Configure all pins as outputs
    gpio.write(GPIO_TRI, 0x00000000)
    print(f"Successfully mapped AXI GPIO at 0x{AXI_GPIO_BASE:08X}")
except Exception as e:
    print(f"Could not map MMIO (Ensure bitstream with AXI GPIO at 0x{AXI_GPIO_BASE:08X} is loaded):", e)



In [ ]:
# Interactive UI Toggle Buttons
btn_red = widgets.ToggleButton(
    value=False,
    description='D5 Red LED (M19)',
    button_style='danger',
    icon='lightbulb-o'
)

btn_green = widgets.ToggleButton(
    value=False,
    description='D6 Green LED (M17)',
    button_style='success',
    icon='lightbulb-o'
)

output = widgets.Output()

def update_leds(change):
    val = 0
    if btn_red.value:
        val |= 0x01
    if btn_green.value:
        val |= 0x02
    try:
        gpio.write(GPIO_DATA, val)
        with output:
            output.clear_output(wait=True)
            print(f"Hardware Register 0x{GPIO_DATA:04X} = 0x{val:02X} | Red: {btn_red.value} | Green: {btn_green.value}")
    except Exception as e:
        with output:
            output.clear_output(wait=True)
            print("MMIO Write error:", e)

btn_red.observe(update_leds, 'value')
btn_green.observe(update_leds, 'value')

display(widgets.HBox([btn_red, btn_green]))
display(output)



In [ ]:
# Automated Strobe / Blink Test
print("Blinking LEDs 5 times...")
try:
    for _ in range(5):
        gpio.write(GPIO_DATA, 0x01) # Red ON
        time.sleep(0.2)
        gpio.write(GPIO_DATA, 0x02) # Green ON
        time.sleep(0.2)
        gpio.write(GPIO_DATA, 0x03) # Both ON
        time.sleep(0.2)
        gpio.write(GPIO_DATA, 0x00) # Both OFF
        time.sleep(0.2)
    print("Blink test complete.")
except Exception as e:
    print("Error during blink test:", e)

